Load the consolidated dataset

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/Preprocessed_Data/consolidated_dataframe.csv")

OUTPUT_DIR = Path("../../data/processed")
REPORT_DIR = Path("../../reports")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (5400, 14)


,District,Year,Month,Cases,YearMonth,Precipitation_mm,Temp_Min_C,Temp_Max_C,Soil_Moisture_0_7cm,MRiceArea,MRiceYield,SRiceArea,SRiceYield,Population
0,Ampara,2007,1,0,2007-01,144.799990,20.8005,28.350500,0.361118,"51,803","4,934","53,787","4,993",613439
1,Anuradhapura,2007,1,6,2007-01,48.000000,18.6780,29.528000,0.286926,"57,562","4,726","14,635","4,804",822222
2,Badulla,2007,1,5,2007-01,59.700000,15.9110,25.061000,0.438316,"22,266","4,434","10,705","4,639",792806
3,Batticaloa,2007,1,0,2007-01,191.900000,21.3000,28.600000,0.317090,"18,469","2,891","2,705","3,680",500109
4,Colombo,2007,1,12,2007-01,62.300003,20.3110,32.260998,0.199657,"3,354","3,033",879,"3,346",2303397


Clean numeric columns and missing-value markers

In [5]:
# Columns that should contain numeric values
numeric_columns = [
    "Year", "Month", "Cases",
    "Precipitation_mm", "Temp_Min_C", "Temp_Max_C",
    "Soil_Moisture_0_7cm",
    "MRiceArea", "MRiceYield",
    "SRiceArea", "SRiceYield",
    "Population"
]

# Work on a copy; preserve the original dataset
clean_df = df.copy()

# Convert common missing-value markers to NaN
missing_markers = ["-", "_", "", "NA", "N/A", "NULL", "None"]

for col in numeric_columns:
    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
        .replace(missing_markers, pd.NA)
        .str.replace(",", "", regex=False)
    )

    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

print("Numeric conversion completed.")
display(clean_df[numeric_columns].isna().sum().to_frame(
    name="Missing_Count"
))

Numeric conversion completed.


,Missing_Count
Year,0
Month,0
Cases,0
Precipitation_mm,0
Temp_Min_C,0
Temp_Max_C,0
Soil_Moisture_0_7cm,0
MRiceArea,36
MRiceYield,36
SRiceArea,432


Create the monthly date variable and select 2008–2024

In [6]:
clean_df["Date"] = pd.to_datetime(
    clean_df["YearMonth"].astype("string").str.strip(),
    format="%Y-%m",
    errors="coerce"
)

# Check date consistency before filtering
date_mismatch = (
    (clean_df["Date"].dt.year != clean_df["Year"]) |
    (clean_df["Date"].dt.month != clean_df["Month"])
)

print("Invalid dates:", clean_df["Date"].isna().sum())
print("Date inconsistencies:", date_mismatch.sum())

# Keep the main study period
clean_df = clean_df[
    clean_df["Date"].between("2008-01-01", "2024-12-31")
].copy()

clean_df = clean_df.sort_values(
    ["District", "Date"]
).reset_index(drop=True)

print("Filtered dataset shape:", clean_df.shape)
print("Start date:", clean_df["Date"].min())
print("End date:", clean_df["Date"].max())
print("Districts:", clean_df["District"].nunique())

display(clean_df.head())

Invalid dates: 0
Date inconsistencies: 0
Filtered dataset shape: (5100, 15)
Start date: 2008-01-01 00:00:00
End date: 2024-12-01 00:00:00
Districts: 25


,District,Year,Month,Cases,YearMonth,Precipitation_mm,Temp_Min_C,Temp_Max_C,Soil_Moisture_0_7cm,MRiceArea,MRiceYield,SRiceArea,SRiceYield,Population,Date
0,Ampara,2008,1,1,2008-01,177.6,19.8505,29.6005,0.380341,64490,4864,60168,4927,620469,2008-01-01
1,Ampara,2008,2,3,2008-02,121.0,18.9005,28.9005,0.362213,64490,4864,60168,4927,620469,2008-02-01
2,Ampara,2008,3,0,2008-03,293.6,21.8005,30.7505,0.385442,64490,4864,60168,4927,620469,2008-03-01
3,Ampara,2008,4,1,2008-04,119.00001,23.0505,32.5505,0.359476,64490,4864,60168,4927,620469,2008-04-01
4,Ampara,2008,5,4,2008-05,34.600002,24.3005,34.5505,0.269909,64490,4864,60168,4927,620469,2008-05-01


Recheck the monthly panel

In [7]:
# Check whether every district has all 204 months
district_month_counts = (
    clean_df.groupby("District")["Date"]
    .nunique()
    .rename("Observed_Months")
    .reset_index()
)

district_month_counts["Expected_Months"] = 204
district_month_counts["Missing_Months"] = (
    district_month_counts["Expected_Months"]
    - district_month_counts["Observed_Months"]
)

display(district_month_counts)

# Check duplicate district-month combinations
duplicate_count = clean_df.duplicated(
    subset=["District", "Date"]
).sum()

print("Duplicate district-month records:", duplicate_count)

assert clean_df["District"].nunique() == 25
assert len(clean_df) == 5100
assert duplicate_count == 0
assert clean_df["Date"].notna().all()

,District,Observed_Months,Expected_Months,Missing_Months
0,Ampara,204,204,0
1,Anuradhapura,204,204,0
2,Badulla,204,204,0
3,Batticaloa,204,204,0
4,Colombo,204,204,0
5,Galle,204,204,0
6,Gampaha,204,204,0
7,Hambantota,204,204,0
8,Jaffna,204,204,0
9,Kalutara,204,204,0


Duplicate district-month records: 0


Save the cleaned dataset and missing-value report

In [9]:
# Save cleaned data without imputing missing values
cleaned_path = OUTPUT_DIR / "leptospirosis_2008_2024.csv"

clean_df.to_csv(cleaned_path, index=False)

# Create a missing-value report
missing_report = pd.DataFrame({
    "Variable": clean_df.columns,
    "Missing_Count": [
        clean_df[col].isna().sum()
        for col in clean_df.columns
    ],
    "Missing_Percentage": [
        round(clean_df[col].isna().mean() * 100, 2)
        for col in clean_df.columns
    ]
}).sort_values("Missing_Count", ascending=False)

missing_report.to_csv(
    REPORT_DIR / "missing_values_2008_2024.csv",
    index=False
)

display(missing_report)

print("Cleaned dataset saved to:", cleaned_path)

,Variable,Missing_Count,Missing_Percentage
12,SRiceYield,348,6.82
11,SRiceArea,348,6.82
10,MRiceYield,36,0.71
9,MRiceArea,36,0.71
0,District,0,0.00
1,Year,0,0.00
2,Month,0,0.00
6,Temp_Min_C,0,0.00
5,Precipitation_mm,0,0.00
4,YearMonth,0,0.00


Cleaned dataset saved to: ..\..\data\processed\leptospirosis_2008_2024.csv
